In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-terrain2-kp2000kd50-relinvel20'  # ckpt = 20000
# exp_name = 'friction-walking-terrain1-kp4000kd50-linvel20-correct0.1-angvel4-plus-0.5'  # ckpt = 20000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.0172, -0.4736, -0.7030,  0.5600, -1.1907,  0.1431, -0.1918,  0.5651,
          0.6153, -0.0524, -1.3241,  0.2163]], device='cuda:0')
Scaled actions :  tensor([[-0.0172, -0.4736, -0.7030,  0.5600, -1.1907,  0.1431, -0.1918,  0.5651,
          0.6153, -0.0524, -1.3241,  0.2163]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-1.2959e-18,  1.5550e-08,  4.7798e-19,  5.0845e-10,  3.7697e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4362e-08,
          3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04, -3.4389e-08,
         -8.5621e-08, -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,
          7.6633e-07,  7.1809e-07,  1.5422e-06, -7.0859e-03,  1.8227e-02,
         -9.5517e-03, -1.7194e-06, -4.2811e-06, -2.3818e-06, -7.0895e-03,
          1.8232e-02, -9.5524e-03,  3.8316e-05, -1.7207e-02, -4.7359e-01,
         -7.0297e-01,  5.6004e-01, -1.1907e+00,  1.4310e-01, -1.9184e-01,
          5.6513e-01,  6.1531e-01, -5.2368e-02, -1.3241e+00,  2.1630e-01]],
       device='cuda:0')
torques: [-1.58794924e-16 -1.00915112e-15  2.37314235e-06  7.56007923e-06
  1.59905156e-06  6.12617365e-17 -6.48048861e-18 -7.17891685e-16
  2.37314235e-06  7.56007923e-06  1.59905156e-06 -8.75046755e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[ 0.0051, -0.6163, -0.9500,  0.2463, -1.1872,  0.0475, -0.1002,  0.5889,
          0.9504,  0.6572, -1.4729,  0.3976]], device='cuda:0')
Scaled actions :  tensor([[ 0.0051, -0.6163, -0.9500,  0.2463, -1.1872,  0.0475, -0.1002,  0.5889,
          0.9504,  0.6572, -1.4729,  0.3976]], device='cuda:0')
obs :  tensor([[-0.1913, -0.1127,  0.6774, -0.0028,  0.0040, -1.0000,  1.0000,  0.0000,
          0.0000, -0.0063, -0.0038, -0.0198,  0.0382, -0.1079,  0.0430, -0.0368,
         -0.0021,  0.0065,  0.0182, -0.0910,  0.0421, -0.0299, -0.0294, -0.1709,
          0.3248, -0.9717,  0.2227, -0.2845, -0.0067,  0.0806,  0.1228, -0.8123,
          0.2944,  0.0051, -0.6163, -0.9500,  0.2463, -1.1872,  0.0475, -0.1002,
          0.5889,  0.9504,  0.6572, -1.4729,  0.3976]], device='cuda:0')
torques: [   9.93104125 -200.         -200.          200.         -200.
  -22.31676903  -47.29244868  200.          200.         -200.
 -200.           65.24784826]
データ収集: step 3


In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.0867, -0.4538, -0.0877, -0.4838,  0.2649, -0.2285,  0.5296, -0.5480,
          0.1143,  0.0342, -0.4285,  0.2802]], device='cuda:0')
Scaled actions :  tensor([[-0.0867, -0.4538, -0.0877, -0.4838,  0.2649, -0.2285,  0.5296, -0.5480,
          0.1143,  0.0342, -0.4285,  0.2802]], device='cuda:0')
obs :  tensor([[-2.2638e-01, -1.1425e-01,  7.6690e-01, -7.4056e-03,  1.2853e-02,
         -9.9989e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -3.7454e-03,
         -1.6306e-02, -6.3630e-02,  1.1354e-01, -3.7851e-01,  4.8779e-02,
         -7.1270e-02,  7.4212e-04,  2.5705e-02,  6.0279e-02, -3.5962e-01,
          1.5219e-01,  2.7284e-02, -9.7230e-02, -2.4738e-01,  3.7963e-01,
         -1.6567e+00,  1.6251e-03, -1.0863e-01,  4.0133e-02,  1.1181e-01,
          2.6227e-01, -1.7861e+00,  5.3121e-01, -8.6746e-02, -4.5384e-01,
         -8.7741e-02, -4.8379e-01,  2.6488e-01, -2.2855e-01,  5.2964e-01,
         -5.4797e-01,  1.1429e-01,  3.4214e-02, -4.2849e-01,  2.8

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.3411,  0.0611,  0.1745, -0.6664,  0.6846,  0.1692,  0.4388,  0.3275,
          0.3204,  0.0119, -0.0755,  0.4834]], device='cuda:0')
Scaled actions :  tensor([[-0.3411,  0.0611,  0.1745, -0.6664,  0.6846,  0.1692,  0.4388,  0.3275,
          0.3204,  0.0119, -0.0755,  0.4834]], device='cuda:0')
obs :  tensor([[ 0.3923, -0.2233,  0.5155, -0.0157,  0.0086, -0.9998,  1.0000,  0.0000,
          0.0000, -0.0223, -0.0609, -0.0872,  0.1516, -0.6044, -0.0254, -0.0387,
         -0.0186,  0.0612,  0.0885, -0.6105,  0.2030, -0.1425, -0.3338, -0.0468,
          0.0465, -0.6977, -0.4375,  0.3894, -0.2037,  0.1989,  0.0586, -0.8180,
          0.1772, -0.3411,  0.0611,  0.1745, -0.6664,  0.6846,  0.1692,  0.4388,
          0.3275,  0.3204,  0.0119, -0.0755,  0.4834]], device='cuda:0')
torques: [-139.11239874 -200.           53.3134535  -200.          200.
   28.61111717  200.         -200.          -83.67527936 -179.30357134
  200.          -24.02661637]
データ収集:

In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[ 0.2117,  1.1893, -1.1269, -0.1983, -0.6199,  0.3006, -0.2373,  0.5186,
         -0.1495, -0.4819, -0.6606,  0.4330]], device='cuda:0')
Scaled actions :  tensor([[ 0.2117,  1.1893, -1.1269, -0.1983, -0.6199,  0.3006, -0.2373,  0.5186,
         -0.1495, -0.4819, -0.6606,  0.4330]], device='cuda:0')
obs :  tensor([[-0.1029, -0.6491,  0.5546, -0.0353,  0.0048, -0.9994,  1.0000,  0.0000,
          0.0000, -0.0967, -0.1002, -0.0670,  0.1310, -0.6401, -0.0135,  0.0896,
         -0.0283,  0.1353,  0.0730, -0.6686,  0.2898, -0.4887, -0.0987,  0.2042,
         -0.2210,  0.2465,  0.3615,  0.7494,  0.0944,  0.4381, -0.1298,  0.1457,
          0.4194,  0.2117,  1.1893, -1.1269, -0.1983, -0.6199,  0.3006, -0.2373,
          0.5186, -0.1495, -0.4819, -0.6606,  0.4330]], device='cuda:0')
torques: [ -24.3716787   200.          200.         -200.          200.
   31.63918516  -24.86772631  200.          -48.17576508   -1.02044658
  200.          -33.01389307]
データ収集:

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.9947, -0.9014, -1.4474,  0.8321,  0.1095, -0.1247, -0.9504, -1.4449,
         -0.8833, -1.1342,  0.2665, -0.6195]], device='cuda:0')
Scaled actions :  tensor([[ 0.9947, -0.9014, -1.4474,  0.8321,  0.1095, -0.1247, -0.9504, -1.4449,
         -0.8833, -1.1342,  0.2665, -0.6195]], device='cuda:0')
obs :  tensor([[-0.7251, -0.0464,  0.6784, -0.0476,  0.0234, -0.9986,  1.0000,  0.0000,
          0.0000, -0.1493, -0.0959, -0.0407,  0.0747, -0.6294,  0.0873,  0.1904,
          0.0208,  0.1804,  0.0554, -0.6699,  0.3371, -0.0796,  0.1303,  0.0689,
         -0.3119,  0.0134,  0.4218,  0.3218,  0.3345,  0.0545, -0.0546,  0.0020,
          0.1704,  0.9947, -0.9014, -1.4474,  0.8321,  0.1095, -0.1247, -0.9504,
         -1.4449, -0.8833, -1.1342,  0.2665, -0.6195]], device='cuda:0')
torques: [ 200.          200.         -200.         -200.           13.85324106
   41.40447049 -200.          200.         -200.         -200.
   22.80031943   18.73921524]
データ収集:

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.2769, -0.4349,  0.4065,  0.2152,  0.0335,  0.2331, -0.1583, -0.6559,
          1.3250, -0.1772, -0.4240, -0.0977]], device='cuda:0')
Scaled actions :  tensor([[ 0.2769, -0.4349,  0.4065,  0.2152,  0.0335,  0.2331, -0.1583, -0.6559,
          1.3250, -0.1772, -0.4240, -0.0977]], device='cuda:0')
obs :  tensor([[-0.3538,  0.5057,  0.5233, -0.0376,  0.0450, -0.9983,  1.0000,  0.0000,
          0.0000, -0.1179, -0.0817, -0.0737,  0.0948, -0.5129,  0.0948,  0.1994,
          0.0612,  0.1674,  0.0454, -0.5684,  0.2615,  0.3695, -0.0140, -0.3225,
          0.2419,  0.8803, -0.3582, -0.1868,  0.0975, -0.1577, -0.0586,  0.9321,
         -0.8160,  0.2769, -0.4349,  0.4065,  0.2152,  0.0335,  0.2331, -0.1583,
         -0.6559,  1.3250, -0.1772, -0.4240, -0.0977]], device='cuda:0')
torques: [ 200.         -200.         -200.          200.          200.
 -179.37998676 -200.         -200.         -200.         -200.
  200.         -200.        ]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[-0.7788,  0.1377, -0.3335, -0.0539, -1.4300,  0.2605,  0.3606, -0.3125,
          0.9459, -1.1972, -0.0803,  0.0170]], device='cuda:0')
Scaled actions :  tensor([[-0.7788,  0.1377, -0.3335, -0.0539, -1.4300,  0.2605,  0.3606, -0.3125,
          0.9459, -1.1972, -0.0803,  0.0170]], device='cuda:0')
obs :  tensor([[ 0.1441, -0.1519,  0.5895, -0.0306,  0.0493, -0.9983,  1.0000,  0.0000,
          0.0000,  0.0032, -0.1209, -0.1020,  0.1336, -0.3097,  0.1167,  0.1094,
          0.0597,  0.1691,  0.0254, -0.4150,  0.1264,  0.6692, -0.3210,  0.0255,
          0.1404,  0.8714,  0.2519, -0.5575, -0.0996,  0.1206, -0.1001,  0.3999,
         -0.5225, -0.7788,  0.1377, -0.3335, -0.0539, -1.4300,  0.2605,  0.3606,
         -0.3125,  0.9459, -1.1972, -0.0803,  0.0170]], device='cuda:0')
torques: [-119.77659637 -200.          200.           15.75596639 -200.
  -14.07560172  -10.75948605 -200.          200.         -200.
 -200.           79.7024128 ]
データ収集: step 9


In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-1.0249,  0.7048, -1.1857,  0.2572, -1.2933, -0.1234,  1.4190, -0.0871,
         -0.8716, -1.1059, -0.7635, -0.1185]], device='cuda:0')
Scaled actions :  tensor([[-1.0249,  0.7048, -1.1857,  0.2572, -1.2933, -0.1234,  1.4190, -0.0871,
         -0.8716, -1.1059, -0.7635, -0.1185]], device='cuda:0')
obs :  tensor([[ 0.1364, -0.1868,  0.7889, -0.0361,  0.0446, -0.9984,  1.0000,  0.0000,
          0.0000,  0.0696, -0.1690, -0.0961,  0.1443, -0.2393,  0.1849,  0.0417,
          0.0374,  0.1940,  0.0021, -0.2985,  0.0717,  0.0541, -0.1708,  0.0178,
         -0.0152, -0.0740,  0.1640, -0.1730, -0.1101,  0.1524, -0.1581,  0.7196,
         -0.3039, -1.0249,  0.7048, -1.1857,  0.2572, -1.2933, -0.1234,  1.4190,
         -0.0871, -0.8716, -1.1059, -0.7635, -0.1185]], device='cuda:0')
torques: [-200.          200.         -200.         -200.         -200.
 -162.47956662  200.         -200.          200.         -200.
  200.         -131.25764528]
データ収集: step 10

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[ 0.4822, -0.5389, -0.5362,  0.9488, -0.7020, -0.2069,  0.0774, -0.4530,
          0.9658, -0.6157, -0.2762, -0.1952]], device='cuda:0')
Scaled actions :  tensor([[ 0.4822, -0.5389, -0.5362,  0.9488, -0.7020, -0.2069,  0.0774, -0.4530,
          0.9658, -0.6157, -0.2762, -0.1952]], device='cuda:0')
obs :  tensor([[-2.8588e-02,  3.4366e-01,  6.4831e-01, -3.0490e-02,  4.4069e-02,
         -9.9856e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.6504e-02,
         -1.8089e-01, -1.3559e-01,  1.6617e-01, -3.5946e-01,  1.1933e-01,
          4.7880e-02,  2.6005e-02,  1.9851e-01, -1.5490e-02, -2.4722e-01,
         -6.5112e-03, -5.2643e-01,  3.3581e-02, -3.5796e-01,  1.7410e-01,
         -1.0306e+00, -5.3633e-01,  1.6862e-01, -5.1455e-04, -9.9886e-02,
          3.9752e-02, -9.2871e-02, -2.3656e-01,  4.8216e-01, -5.3894e-01,
         -5.3618e-01,  9.4876e-01, -7.0204e-01, -2.0694e-01,  7.7431e-02,
         -4.5300e-01,  9.6578e-01, -6.1573e-01, -2.7623e-01, -1.

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.8714,  0.4863,  0.2312, -0.2698, -0.1835,  0.3687, -0.0912,  0.2645,
          0.5784, -0.7432, -0.2165,  0.0251]], device='cuda:0')
Scaled actions :  tensor([[ 0.8714,  0.4863,  0.2312, -0.2698, -0.1835,  0.3687, -0.0912,  0.2645,
          0.5784, -0.7432, -0.2165,  0.0251]], device='cuda:0')
obs :  tensor([[ 0.2779,  0.1970,  0.4479, -0.0191,  0.0389, -0.9991,  1.0000,  0.0000,
          0.0000, -0.0367, -0.1973, -0.2111,  0.2254, -0.5422,  0.0039,  0.0595,
          0.0161,  0.1921, -0.0079, -0.2580, -0.0585, -0.0540, -0.1881, -0.3970,
          0.3946, -0.8168, -0.5740, -0.0059, -0.0909,  0.0106,  0.0363, -0.0425,
         -0.2280,  0.8714,  0.4863,  0.2312, -0.2698, -0.1835,  0.3687, -0.0912,
          0.2645,  0.5784, -0.7432, -0.2165,  0.0251]], device='cuda:0')
torques: [ 200.         -200.         -200.          200.         -200.
 -200.           37.54290192 -200.          200.         -200.
    4.27284495  -46.55412413]
データ収集: step 1

In [48]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=1.023, Scaled action max=1.023
Step 1/10, Total steps: 232
steps: 232
actions : tensor([[-1.0134, -0.8189, -0.7600,  1.0230,  0.0152, -0.0177,  0.1002, -0.3300,
          0.2876, -0.4907,  0.2302, -0.6385]], device='cuda:0')
target_dof_pos: tensor([[-0.9271, -0.5633, -2.7935,  2.4689, -0.8236,  0.4388,  0.4499, -0.2447,
         -1.5454,  0.8927, -0.8061, -0.6337]], device='cuda:0')
Step 1: Original action max=1.295, Scaled action max=1.295
Step 2: Original action max=1.800, Scaled action max=1.800
データ収集完了: 10 steps collected with action_scale=1.0


In [49]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [50]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
